In [2]:
# pip install -U bitsandbytes>=0.46.1

In [4]:
# ── Cell 0: Install dependencies ─────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-U", "bitsandbytes>=0.46.1"], check=True)
print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.2 MB/s eta 0:00:00
Dependencies installed


In [5]:
# ── Cell 1: Auth (always run first) ─────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))
print("Authenticated")

# ── Cell 2: Imports ──────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import pandas as pd

# ── Cell 3: Load model (takes ~30s from cache) ───────────────────────────────
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,
)
print(f"Model loaded — Device: {next(model.parameters()).device}")

Authenticated


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded — Device: cuda:0


In [6]:
def build_zero_shot_prompt(article_text, stance_label=None):
    """
    Single article input. stance_label is optional — only include
    if you decide to tell the model the political leaning.
    """
    if stance_label:
        source_description = f"The following is a {stance_label} news article."
    else:
        source_description = "The following is a news article."
    
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a neutral news summarizer. Your task is to generate a balanced, \
factual summary of the provided news article without introducing bias or \
favoring any political perspective.<|eot_id|><|start_header_id|>user<|end_header_id|>
{source_description}

ARTICLE:
{article_text[:800]}

Write a neutral, factual summary (3-5 sentences) of this article.<|eot_id|>\
<|start_header_id|>assistant<|end_header_id|>
"""


def generate_summary(prompt, model, tokenizer, max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,      # near-deterministic for reproducibility
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the newly generated tokens, not the prompt
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/imanghotbi/allside/allsides_processed_with_splits.csv")

test_df = df[df["split"] == "test"].reset_index(drop=True)
val_df = test_df.sample(n=20, random_state=42).reset_index(drop=True)



results = []
for i, (_, row) in enumerate(test_df.iterrows()):
    for stance in ["left", "center", "right"]:
        article_text = str(row[f"{stance}_text"])
        prompt = build_zero_shot_prompt(article_text)
        summary = generate_summary(prompt, model, tokenizer)
        
        results.append({
            "example_id":        i,
            "issue":             row["issue"],
            "topic":             row["topic"],
            "stance":            stance,
            "input_article":     article_text,
            "generated_summary": summary,
            "roundup_text":      row["roundup_text"],
            "valence":           row.get(f"{stance}_valence"),
            "arousal":           row.get(f"{stance}_arousal"),
        })
    
    if (i + 1) % 10 == 0:
        pd.DataFrame(results).to_csv("/kaggle/working/zero_shot_summaries.csv", index=False)
        print(f"Saved {i+1}/{len(test_df)}")

pd.DataFrame(results).to_csv("/kaggle/working/zero_shot_summaries.csv", index=False)
print(f"Done. Total rows: {len(results)}")  # should be 307 × 3 = 921
print("Saved.")



Saved 10/307
Saved 20/307
Saved 30/307
Saved 40/307
Saved 50/307
Saved 60/307
Saved 70/307
Saved 80/307
Saved 90/307
Saved 100/307
Saved 110/307
Saved 120/307
Saved 130/307
Saved 140/307
Saved 150/307
Saved 160/307
Saved 170/307
Saved 180/307
Saved 190/307
Saved 200/307
Saved 210/307
Saved 220/307
Saved 230/307
Saved 240/307
Saved 250/307
Saved 260/307
Saved 270/307
Saved 280/307
Saved 290/307
Saved 300/307
Done. Total rows: 921
Saved.
